# Dataset por distrito: crimes + IPTU (picos) + receita Airbnb

Gera **dois** datasets, uma linha por distrito (96), com as mesmas colunas:

| arquivo | `iptu_venal_m2` |
|---|---|
| `data/processed/distrito_crime_iptu_receita_pico1.csv` | 1º pico (modo baixo) do valor venal/m² se o distrito for multimodal; senão a mediana |
| `data/processed/distrito_crime_iptu_receita_pico2.csv` | 2º pico (modo alto) se for multimodal; senão a mediana |

- **IPTU**: mesmo cálculo e mesmo critério de multimodalidade do `initialAnalysis/distribuicao_valor_m2_distrito.ipynb`
  (≥ 2 picos no KDE com proeminência ≥ 10% **e** ≥ 10% dos imóveis de cada lado do vale entre os dois picos mais
  proeminentes). Os picos são as posições desses dois máximos do KDE (o modo de fato). O critério antigo, com GMM
  e D de Ashman > 2, só achava 6 distritos; este acha 20.
- **Receita**: mediana de `estimated_revenue_l365d` por bairro (`neighbourhood_cleansed`). A receita não teve
  bairro multimodal (`initialAnalysis/distribuicao_receita_airbnb_bairro.ipynb`), então ela é a mesma nos dois arquivos.
- **Crimes**: colunas `crime_*` do `distrito_features.csv` (gerado pelo `aggregation.ipynb`, com o crosswalk DP → distrito revisado).

## lib

In [1]:
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

warnings.filterwarnings("ignore")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "src").is_dir())
PROC, EXT = ROOT / "data" / "processed", ROOT / "data" / "external"

def chave(s):
    """Nome de distrito/bairro sem acento e em maiúsculas: o Airbnb escreve "Sao Rafael", o IPTU "SAO RAFAEL"."""
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().upper().strip()

## 1. IPTU: valor venal/m² por distrito

Cópia do cálculo de `distribuicao_valor_m2_distrito.ipynb` (seção de dados e seção 3).

In [2]:
COLS = ["numero_contribuinte", "finalidade_imovel", "valor_terreno", "valor_construcao",
        "area_terreno", "area_construida", "fracao_ideal", "fator_obsolescencia"]
it = pd.read_csv(PROC / "IPTU" / "iptu_2025_clean.csv", usecols=COLS, dtype={"numero_contribuinte": str})

# imóvel -> distrito pela quadra fiscal (6 primeiros dígitos do SQL)
q2d = pd.read_csv(EXT / "quadra_to_distrito.csv", dtype={"sq": str}).set_index("sq")["distrito"]
it["distrito"] = it["numero_contribuinte"].str.zfill(11).str[:6].map(q2d)

RES = ["Residência", "Apartamento Em Condomínio",
       "Residência E Outro Uso (Predominância Residencial)",
       "Residência Coletiva, Exclusive Cortiço (Mais De Uma Residência No Lote)",
       "Prédio De Apartamento, Não Em Condomínio, De Uso Exclusivamente Residencial",
       "Flat Residencial Em Condomínio"]
r = it[it["finalidade_imovel"].isin(RES) & it["distrito"].notna() & (it["area_construida"] > 0)].copy()
del it

r["venal_m2"] = (r["area_terreno"] * r["fracao_ideal"] * r["valor_terreno"]
                 + r["area_construida"] * r["valor_construcao"] * r["fator_obsolescencia"]) / r["area_construida"]
lo, hi = r["venal_m2"].quantile([0.001, 0.999])
r = r[r["venal_m2"].between(lo, hi)]
r["log_m2"] = np.log10(r["venal_m2"])
print(f"imóveis residenciais: {len(r):,} | distritos: {r['distrito'].nunique()}")

imóveis residenciais: 3,020,176 | distritos: 96


In [3]:
# mesmos parâmetros e mesma ordem de sorteio do notebook de distribuição (primeiro os KDEs, depois o
# diagnóstico): assim a subamostra de 20 mil é idêntica e os distritos multimodais são os mesmos
GRID = np.linspace(r["log_m2"].min(), r["log_m2"].max(), 600)
rng = np.random.default_rng(42)
AMOSTRA = 20_000
PROEMINENCIA, PESO_MIN = 0.10, 0.10

def amostra(x):
    return x if len(x) <= AMOSTRA else rng.choice(x, AMOSTRA, replace=False)

for d, g in r.groupby("distrito")["log_m2"]:   # só consome o rng na mesma sequência do notebook original
    gaussian_kde(amostra(g.values))(GRID)

def diagnostico(x):
    xs = amostra(x)
    dens = gaussian_kde(xs)(GRID)
    picos, props = find_peaks(dens, prominence=PROEMINENCIA * dens.max())
    if len(picos) < 2:
        return pd.Series({"n_picos": len(picos), "pico1": np.nan, "pico2": np.nan, "massa_menor": np.nan})
    # os 2 picos mais proeminentes; o vale é o ponto mais baixo do KDE entre eles
    p1, p2 = np.sort(picos[np.argsort(props["prominences"])[-2:]])
    vale = GRID[p1 + np.argmin(dens[p1:p2 + 1])]
    frac = (x < vale).mean()   # massa em todos os imóveis do distrito
    return pd.Series({"n_picos": len(picos), "pico1": 10**GRID[p1], "pico2": 10**GRID[p2],
                      "massa_menor": min(frac, 1 - frac)})

diag = r.groupby("distrito")["log_m2"].apply(lambda s: diagnostico(s.values)).unstack()
diag["mediana"] = r.groupby("distrito")["venal_m2"].median()
diag["multimodal"] = (diag["n_picos"] >= 2) & (diag["massa_menor"] >= PESO_MIN)

# 1º pico = modo baixo, 2º pico = modo alto; distrito não multimodal fica com a mediana nos dois
diag["pico1"] = np.where(diag["multimodal"], diag["pico1"], diag["mediana"])
diag["pico2"] = np.where(diag["multimodal"], diag["pico2"], diag["mediana"])

print(f"multimodais: {diag['multimodal'].sum()} de {len(diag)}")
diag.loc[diag["multimodal"], ["n_picos", "massa_menor", "mediana", "pico1", "pico2"]].round(2)

multimodais: 20 de 96


,n_picos,massa_menor,mediana,pico1,pico2
distrito,,,,,
ALTO DE PINHEIROS,2.0,0.39,4304.84,4050.08,5357.50
ARTUR ALVIM,2.0,0.47,1905.76,1226.76,2549.91
BARRA FUNDA,2.0,0.36,2891.38,2779.13,3598.00
BELA VISTA,2.0,0.36,2830.99,2719.96,4138.19
BOM RETIRO,2.0,0.42,2374.87,1418.56,2705.37
CAMBUCI,2.0,0.42,2875.40,2854.90,3617.41
CANGAIBA,2.0,0.42,2126.81,2045.17,2509.09
CONSOLACAO,2.0,0.44,3077.28,2549.91,4050.08
CURSINO,2.0,0.46,2995.19,2468.91,3409.54


## 2. Receita Airbnb: mediana por bairro

In [4]:
a = pd.read_csv(PROC / "Airbnb" / "listings_clean.csv", usecols=["neighbourhood_cleansed", "estimated_revenue_l365d"])
# mesmo recorte do notebook de distribuição: sem receita zero (anúncio inativo) e sem os 0,1% extremos
a = a[a["estimated_revenue_l365d"] > 0]
lo, hi = a["estimated_revenue_l365d"].quantile([0.001, 0.999])
a = a[a["estimated_revenue_l365d"].between(lo, hi)]
a["k"] = a["neighbourhood_cleansed"].map(chave)

receita = a.groupby("k")["estimated_revenue_l365d"].agg(receita_mediana="median", receita_n="size")
print(f"bairros com receita: {len(receita)} | com menos de 30 anúncios: {(receita['receita_n'] < 30).sum()}")

bairros com receita: 95 | com menos de 30 anúncios: 41


## 3. Crimes e montagem

In [5]:
feat = pd.read_csv(PROC / "distrito_features.csv")
# contagens de crime (2025, por tipo e agregadas, mais o total de 2024); crime_source e crime_n_dp são metadado
CRIME = [c for c in feat.columns if c.startswith("crime_") and c not in ("crime_source", "crime_n_dp")]
base = feat[["distrito", "regiao5", "area_km2", *CRIME]].copy()
base["k"] = base["distrito"].map(chave)

base = base.join(diag[["mediana", "pico1", "pico2", "multimodal"]]
                 .rename(columns={"mediana": "iptu_venal_m2_mediana", "multimodal": "iptu_multimodal"}),
                 on="distrito")
base = base.join(receita, on="k").drop(columns="k")
base["receita_n"] = base["receita_n"].fillna(0).astype(int)

assert base["iptu_venal_m2_mediana"].notna().all(), "distrito sem IPTU"
print("distritos sem receita:", base.loc[base["receita_mediana"].isna(), "distrito"].tolist())

COLS_SAIDA = ["distrito", "regiao5", "area_km2", "iptu_venal_m2", "iptu_venal_m2_mediana", "iptu_multimodal",
              "receita_mediana", "receita_n", *CRIME]
for pico in ("pico1", "pico2"):
    out = base.rename(columns={pico: "iptu_venal_m2"})[COLS_SAIDA].sort_values("distrito")
    caminho = PROC / f"distrito_crime_iptu_receita_{pico}.csv"
    out.to_csv(caminho, index=False)
    print(f"{caminho.relative_to(ROOT)}: {out.shape}")

distritos sem receita: ['IGUATEMI']
data/processed/distrito_crime_iptu_receita_pico1.csv: (96, 28)
data/processed/distrito_crime_iptu_receita_pico2.csv: (96, 28)


In [6]:
# conferência: os dois arquivos só diferem em iptu_venal_m2, e só nos distritos multimodais
p1 = pd.read_csv(PROC / "distrito_crime_iptu_receita_pico1.csv")
p2 = pd.read_csv(PROC / "distrito_crime_iptu_receita_pico2.csv")
dif = p1["iptu_venal_m2"] != p2["iptu_venal_m2"]
assert (dif == p1["iptu_multimodal"]).all()
assert p1.drop(columns="iptu_venal_m2").equals(p2.drop(columns="iptu_venal_m2"))
pd.DataFrame({"distrito": p1.loc[dif, "distrito"], "pico1": p1.loc[dif, "iptu_venal_m2"],
              "mediana": p1.loc[dif, "iptu_venal_m2_mediana"], "pico2": p2.loc[dif, "iptu_venal_m2"]}).round(0)

,distrito,pico1,mediana,pico2
1,ALTO DE PINHEIROS,4050.0,4305.0,5357.0
4,ARTUR ALVIM,1227.0,1906.0,2550.0
5,BARRA FUNDA,2779.0,2891.0,3598.0
6,BELA VISTA,2720.0,2831.0,4138.0
8,BOM RETIRO,1419.0,2375.0,2705.0
13,CAMBUCI,2855.0,2875.0,3617.0
17,CANGAIBA,2045.0,2127.0,2509.0
25,CONSOLACAO,2550.0,3077.0,4050.0
26,CURSINO,2469.0,2995.0,3410.0
30,GUAIANASES,913.0,1802.0,1907.0
